# Petrol override test (inference only, no retraining)

**Question:** is the flattened Diwali peak in the XGB-features run caused by the carried-forward petrol value (z about 3.75)?

**Method:** load the trained model once, forecast the same sample of series several times, and change only the petrol column inside the forecast window each time. Everything else (history, calendar, spend, weights) is identical, so any change in the forecast is caused by petrol.

Run it from the same folder as the prediction notebook (it needs `column_roles.json`, `run_metadata.json`, `series_cache/` and the model checkpoint).

In [ ]:
import os, json, collections.abc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt

from darts import TimeSeries
from darts.models import TFTModel
from pytorch_lightning import Callback

BASE      = os.getcwd()
CACHE_DIR = os.path.join(BASE, "series_cache")
OUT_DIR   = os.path.join(BASE, "petrol_override_test")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------- settings ----------------
N_SAMPLE   = 2000      # series used for the test; None = all (slow). Same series in every scenario, so the comparison is paired.
SEED       = 0
BATCH_SIZE = 512
PETROL_COL = None      # None = find the one shared column with "PETROL" in its name
DOSE_Z     = [0.0, 1.0, 2.0, 3.0]    # petrol z-scores to try (0 = the training average)

In [ ]:
with open(os.path.join(BASE, "column_roles.json")) as f:
    ROLES = json.load(f)
with open(os.path.join(BASE, "run_metadata.json")) as f:
    META = json.load(f)

time_col, target_col, FREQ = ROLES["time_col"], ROLES["target_col"], ROLES["freq"]
static_covariates = ROLES["static_covariates"]
series_covariates = ROLES["series_covariates"]
future_covariates = ROLES["future_covariates"]
INPUT_CHUNK_LENGTH = ROLES["input_chunk_length"]
FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END   = pd.Timestamp(ROLES["forecast_end"])
HORIZON = (FORECAST_END - FORECAST_START).days + 1
FC_DAYS = pd.date_range(FORECAST_START, FORECAST_END, freq=FREQ)
MODEL_NAME = META["model_name"]
safe_name = lambda k: str(k).replace("<>", "_").replace("/", "_").replace("\\", "_")

# Darts pickles the training callbacks inside the saved model. Importing tft_checks and defining an inert
# TBDiagnostics lets a fresh kernel resolve those names (same trick as the prediction notebook).
try:
    import tft_checks  # noqa: F401
except ImportError:
    print("tft_checks not importable; ok unless loading fails on EmbeddingGate")
class TBDiagnostics(Callback):
    def __init__(self, *args, **kwargs):
        pass

model = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)
assert 0 < HORIZON <= model.output_chunk_length, "horizon must be <= output_chunk_length for this test"
print(f"Loaded best checkpoint {MODEL_NAME} | horizon {HORIZON} days | {FORECAST_START.date()} -> {FORECAST_END.date()}")

In [ ]:
# ---------------- series sample ----------------
series_index = pd.read_parquet(os.path.join(CACHE_DIR, "series_index.parquet"))
sel = series_index[series_index["in_forecast"]]
if N_SAMPLE and N_SAMPLE < len(sel):
    sel = sel.sample(n=N_SAMPLE, random_state=SEED)
keys     = sel["key"].tolist()
statics  = [pd.DataFrame([row], columns=static_covariates, dtype=np.float32)
            for row in sel[static_covariates].to_numpy()]
path_ids = sel["spend_path"].to_numpy()
print(f"Series in the test: {len(keys):,}")

# ---------------- calendar and spend ----------------
SHARED_COV  = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
SPEND_PATHS = np.load(os.path.join(CACHE_DIR, "spend_paths.npy"))
CAL_INDEX   = SHARED_COV.time_index
CAL_VALUES  = SHARED_COV.values(copy=True)
shared_cols = list(SHARED_COV.components)
COV_COLUMNS = shared_cols + list(series_covariates)
assert COV_COLUMNS == future_covariates, "covariate order differs from column_roles.json"
assert CAL_INDEX[0] <= FORECAST_START - pd.Timedelta(days=INPUT_CHUNK_LENGTH) and CAL_INDEX[-1] >= FORECAST_END

if PETROL_COL is None:
    cands = [c for c in shared_cols if "PETROL" in c.upper()]
    assert len(cands) == 1, f"expected exactly one petrol column, found {cands}; set PETROL_COL by hand"
    PETROL_COL = cands[0]
pi  = shared_cols.index(PETROL_COL)
fi0 = CAL_INDEX.get_loc(FORECAST_START)
fi1 = CAL_INDEX.get_loc(FORECAST_END)
BASE_Z = CAL_VALUES[fi0:fi1 + 1, pi].copy()
print(f"{PETROL_COL} in the forecast window (z): min {BASE_Z.min():.2f}, max {BASE_Z.max():.2f}, "
      f"{len(np.unique(np.round(BASE_Z, 4)))} distinct value(s)")
tr_end = pd.Timestamp(ROLES["train_end"])
_tr = CAL_VALUES[:CAL_INDEX.get_loc(tr_end) + 1, pi]
print(f"{PETROL_COL} in training (z): min {_tr.min():.2f}, max {_tr.max():.2f}, mean {_tr.mean():.2f}")

# ---------------- Diwali alignment with last year ----------------
d_flag = pd.Series(CAL_VALUES[:, shared_cols.index("D")], index=CAL_INDEX)
def diwali(year):
    days = d_flag[(d_flag.index.year == year) & (d_flag.index.month >= 9) & (d_flag != 0)].index
    assert len(days) == 1, f"expected one D day in {year}, found {len(days)}"
    return days[0]
D_NOW, D_LY = diwali(FORECAST_START.year), diwali(FORECAST_START.year - 1)
SHIFT = D_NOW - D_LY
LY_DAYS = FC_DAYS - SHIFT
print(f"Diwali {D_LY.date()} (last year) vs {D_NOW.date()} (this year): shift {SHIFT.days} days")
Z_LAST_YEAR = CAL_VALUES[[CAL_INDEX.get_loc(d) for d in LY_DAYS], pi]
print(f"Petrol z on the same festival-aligned days last year: {Z_LAST_YEAR.min():.2f} to {Z_LAST_YEAR.max():.2f}")

# ---------------- history (what the encoder reads) and last year's actuals ----------------
HIST, LY_ACTUAL = [], np.zeros(HORIZON)
for i, k in enumerate(keys):
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
        sales, start = z["val_sales"], str(z["val_start"])
    times = pd.date_range(start, periods=len(sales), freq=FREQ)
    assert times[-1] == FORECAST_START - pd.Timedelta(days=1), f"history of {k} ends {times[-1]}"
    assert len(sales) >= INPUT_CHUNK_LENGTH
    HIST.append(TimeSeries.from_times_and_values(times, sales.reshape(-1, 1).astype(np.float32),
                                                 columns=[target_col], static_covariates=statics[i]))
    LY_ACTUAL += pd.Series(sales, index=times).reindex(LY_DAYS).fillna(0).to_numpy()
print(f"Last-year actual total over the aligned window (sample): {LY_ACTUAL.sum():,.0f}")

In [ ]:
# ---------------- forecast with a chosen petrol path ----------------
def build_cov(petrol_z):
    vals = CAL_VALUES.copy()
    vals[fi0:fi1 + 1, pi] = petrol_z          # ONLY the forecast window changes; the encoder history is untouched
    cache, out = {}, []
    for pid in path_ids:
        pid = int(pid)
        if pid not in cache:
            v = np.concatenate([vals, SPEND_PATHS[pid]], axis=1).astype(np.float32)
            cache[pid] = TimeSeries.from_times_and_values(CAL_INDEX, v, columns=COV_COLUMNS)
        out.append(cache[pid])
    return out

EXPECTED = [f"{target_col}_r", f"{target_col}_p"]
def forecast_mean(petrol_z):
    preds = model.predict(n=HORIZON, series=HIST, future_covariates=build_cov(petrol_z),
                          predict_likelihood_parameters=True, num_samples=1,
                          batch_size=BATCH_SIZE, verbose=False)
    mean = np.zeros((len(preds), HORIZON))
    for i, p in enumerate(preds):
        assert list(p.components) == EXPECTED, list(p.components)
        v = p.values(copy=False).astype(np.float64)
        r, pr = v[:, 0], v[:, 1]
        assert np.isfinite(v).all() and (r > 0).all() and ((pr > 0) & (pr < 1)).all()
        mean[i] = r * pr / (1.0 - pr)           # PyTorch NegBin mean, as in the prediction notebook
    return mean

SCENARIOS = {"baseline (as forecast)": BASE_Z}
for z in DOSE_Z:
    SCENARIOS[f"petrol z = {z:+.1f}"] = np.full(HORIZON, z, dtype=np.float32)
SCENARIOS["petrol = same days last year"] = Z_LAST_YEAR

DAILY, t0 = {}, datetime.now()
for name, z in SCENARIOS.items():
    DAILY[name] = forecast_mean(z).sum(axis=0)
    print(f"{name:32s} total {DAILY[name].sum():>12,.0f}   ({(datetime.now() - t0).seconds}s elapsed)")

In [ ]:
# ---------------- report ----------------
dwin = (FC_DAYS >= D_NOW - pd.Timedelta(days=3)) & (FC_DAYS <= D_NOW + pd.Timedelta(days=6))
ly_dwin = float(LY_ACTUAL[dwin].sum())
rows = []
for name, d in DAILY.items():
    s = pd.Series(d, index=FC_DAYS)
    peak_day = s.idxmax()
    rows.append({
        "scenario": name,
        "total": s.sum(),
        "Diwali window (D-3..D+6)": s[dwin].sum(),
        "peak day": peak_day.date(),
        "peak / median day": s.max() / s.median(),
        "vs last-year actual (all days)": s.sum() / LY_ACTUAL.sum(),
        "vs last-year actual (Diwali window)": s[dwin].sum() / ly_dwin,
    })
for m in FC_DAYS.to_period("M").unique():
    for r_, (name, d) in zip(rows, DAILY.items()):
        r_[f"{m.strftime('%b')} share %"] = 100 * pd.Series(d, index=FC_DAYS)[FC_DAYS.to_period("M") == m].sum() / d.sum()
rep = pd.DataFrame(rows).set_index("scenario")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.width", 250)
print(rep.to_string())
print(f"\nLast-year actual on the same series, Diwali window: {ly_dwin:,.0f} | all aligned days: {LY_ACTUAL.sum():,.0f}")

b = rep.loc["baseline (as forecast)", "Diwali window (D-3..D+6)"]
for name in rep.index[1:]:
    print(f"Diwali-window forecast with '{name}' is {rep.loc[name, 'Diwali window (D-3..D+6)'] / b:.2f}x the baseline")

rep.to_csv(os.path.join(OUT_DIR, "petrol_override_summary.csv"))
pd.DataFrame(DAILY, index=FC_DAYS).assign(LAST_YEAR_ACTUAL_ALIGNED=LY_ACTUAL).to_csv(
    os.path.join(OUT_DIR, "petrol_override_daily.csv"), index_label="date")

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(FC_DAYS, LY_ACTUAL, color="black", lw=2.2, label=f"last-year actual (aligned on Diwali, shifted {SHIFT.days}d)")
for name, d in DAILY.items():
    ax.plot(FC_DAYS, d, lw=1.4, ls="--" if name.startswith("baseline") else "-", label=name)
ax.axvline(D_NOW, color="grey", ls=":")
ax.set_title("Same model, same series, only the petrol path changes")
ax.set_ylabel("units per day (sample of series)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "petrol_override.png"), dpi=130)
plt.show()
print("Saved to", OUT_DIR)